# Multi-Agent Systems

## Review

So far, we've built single agents — one LLM with tools in a ReAct loop.

## Goals

Real-world applications often require **multiple specialized agents** working together. We'll explore:

1. **Supervisor pattern** — a coordinator agent routes tasks to specialists
2. **Agent handoff** — one agent delegates to another
3. **Shared state** — agents communicate through graph state

This is the foundation for building complex AI systems where different agents have different capabilities.

![Multi-Agent](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbb2e7fc0c62ac1a3e53f6_multi-agent1.png)

In [ ]:
%run ../langchain_common.py

## Pattern 1: Supervisor Agent

A **supervisor** receives the user request and routes it to the appropriate specialist agent.

```
User → Supervisor → Research Agent
                   → Math Agent
                   → Writing Agent
```

In [ ]:
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from typing import Literal
from typing_extensions import TypedDict, Annotated
from IPython.display import Image, display
import operator

### Define Specialist Tools

Each specialist has access to different tools.

In [ ]:
# Math specialist tool
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. E.g., '2 + 3 * 4'"""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Research specialist tool
@tool
def lookup_fact(topic: str) -> str:
    """Look up a fact about a topic."""
    facts = {
        "python": "Python was created by Guido van Rossum in 1991.",
        "langgraph": "LangGraph is a framework for building stateful multi-agent applications.",
        "earth": "Earth is the third planet from the Sun, with a diameter of 12,742 km.",
    }
    for key, fact in facts.items():
        if key in topic.lower():
            return fact
    return f"No fact found for '{topic}'. Try: python, langgraph, earth"

print("Tools defined: calculate, lookup_fact")

### Define the State

We extend `MessagesState` to track which agent is currently active.

In [ ]:
class MultiAgentState(MessagesState):
    """State shared across all agents."""
    next_agent: str  # Which agent to route to next

### Build the Supervisor

The supervisor decides which specialist should handle the request.

In [ ]:
supervisor_prompt = SystemMessage(content="""You are a supervisor routing requests to specialist agents.
Based on the user's request, respond with ONLY one of these words:
- "math" if the request involves calculations or numbers
- "research" if the request involves facts or knowledge
- "FINISH" if the task is complete and you can respond directly

Respond with ONLY the routing word, nothing else.""")

def supervisor(state: MultiAgentState):
    """Route to the appropriate specialist."""
    response = llm_noreason.invoke([supervisor_prompt] + state["messages"])
    route = response.content.strip().lower()
    
    if "math" in route:
        return {"next_agent": "math_agent"}
    elif "research" in route:
        return {"next_agent": "research_agent"}
    else:
        return {"next_agent": "FINISH", "messages": [response]}

print("Supervisor defined")

### Build Specialist Agents

Each specialist has its own system prompt and tools. They execute tool calls inline and return a final answer.

In [ ]:
# Math agent
math_tools = [calculate]
math_llm = llm_noreason.bind_tools(math_tools)
math_prompt = SystemMessage(content="You are a math specialist. Use the calculate tool to solve math problems. Be concise.")

def math_agent(state: MultiAgentState):
    """Handle math-related requests."""
    response = math_llm.invoke([math_prompt] + state["messages"])
    if response.tool_calls:
        from langchain_core.messages import ToolMessage
        tool_results = []
        for tc in response.tool_calls:
            result = calculate.invoke(tc["args"])
            tool_results.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        final = llm_noreason.invoke([math_prompt] + state["messages"] + [response] + tool_results)
        return {"messages": [final]}
    return {"messages": [response]}

# Research agent
research_tools = [lookup_fact]
research_llm = llm_noreason.bind_tools(research_tools)
research_prompt = SystemMessage(content="You are a research specialist. Use lookup_fact to find information. Be concise.")

def research_agent(state: MultiAgentState):
    """Handle research-related requests."""
    response = research_llm.invoke([research_prompt] + state["messages"])
    if response.tool_calls:
        from langchain_core.messages import ToolMessage
        tool_results = []
        for tc in response.tool_calls:
            result = lookup_fact.invoke(tc["args"])
            tool_results.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        final = llm_noreason.invoke([research_prompt] + state["messages"] + [response] + tool_results)
        return {"messages": [final]}
    return {"messages": [response]}

print("Specialist agents defined")

### Wire the Graph

The supervisor routes to specialists via conditional edges. Specialists return to END.

In [ ]:
def route_from_supervisor(state: MultiAgentState) -> Literal["math_agent", "research_agent", "__end__"]:
    if state["next_agent"] == "FINISH":
        return "__end__"
    return state["next_agent"]

# Build the multi-agent graph
builder = StateGraph(MultiAgentState)

# Add nodes
builder.add_node("supervisor", supervisor)
builder.add_node("math_agent", math_agent)
builder.add_node("research_agent", research_agent)

# Add edges
builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_from_supervisor)
builder.add_edge("math_agent", END)
builder.add_edge("research_agent", END)

# Compile
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### Test the Multi-Agent System

In [ ]:
# Test math routing
result = graph.invoke({"messages": [HumanMessage(content="What is 15 * 23 + 47?")]})
print("=== Math Request ===")
for msg in result["messages"]:
    msg.pretty_print()

In [ ]:
# Test research routing
result = graph.invoke({"messages": [HumanMessage(content="Tell me about Python programming language")]})
print("=== Research Request ===")
for msg in result["messages"]:
    msg.pretty_print()

## Pattern 2: Agent Handoff with Shared State

Sometimes agents need to **collaborate sequentially** — one agent's output feeds into the next.

Example: A research agent finds information, then a writing agent drafts a summary.

In [ ]:
class CollaborativeState(MessagesState):
    """State for sequential agent collaboration."""
    research_notes: str  # Accumulated research
    final_draft: str     # Final written output

def researcher(state: CollaborativeState):
    """Gather information on the topic."""
    prompt = SystemMessage(content="You are a researcher. Provide 3 key facts about the topic. Be brief and factual.")
    response = llm_noreason.invoke([prompt] + state["messages"])
    return {"research_notes": response.content, "messages": [response]}

def writer(state: CollaborativeState):
    """Write a polished summary from research notes."""
    prompt = SystemMessage(content=f"""You are a writer. Using these research notes, write a concise 2-sentence summary.

Research notes: {state['research_notes']}""")
    response = llm_noreason.invoke([prompt] + state["messages"])
    return {"final_draft": response.content, "messages": [response]}

# Build sequential pipeline
collab_builder = StateGraph(CollaborativeState)
collab_builder.add_node("researcher", researcher)
collab_builder.add_node("writer", writer)
collab_builder.add_edge(START, "researcher")
collab_builder.add_edge("researcher", "writer")
collab_builder.add_edge("writer", END)

collab_graph = collab_builder.compile()
display(Image(collab_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = collab_graph.invoke({"messages": [HumanMessage(content="Machine learning")]})

print("=== Research Notes ===")
print(result["research_notes"])
print()
print("=== Final Draft ===")
print(result["final_draft"])

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Supervisor** | One coordinator routes to multiple specialists |
| **Sequential Handoff** | Agents collaborate in a pipeline (research → write) |
| **Shared State** | Agents communicate via typed state fields |

### Design Principles

1. **Single responsibility** — each agent does one thing well
2. **Clear routing logic** — supervisor makes explicit decisions
3. **Typed state** — use state fields for inter-agent communication
4. **Graceful fallback** — supervisor handles unknown requests directly